# <center>Question Answering Machine</center>

In this notebook, we will finetuning BERT model for question answering machine task. We will use facqa dataset from [IndoNLU](https://github.com/indobenchmark/indonlu) and indoBERT model from [indoLEM](https://github.com/indolem).

In [1]:
# !pip install sentencepiece==0.1.95
# !pip install transformers==4.2.2
# !pip install datasets==1.2.0

## Import library

In [1]:
import copy
from datasets import load_dataset
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import numpy as np
from tqdm.auto import tqdm

# if there is a tqdm related error, run this cell one more time

## Load data

In [13]:
# # download facqa dataset from indoNLU repo

# # download train dataset
# !wget "https://raw.githubusercontent.com/indobenchmark/indonlu/master/dataset/facqa_qa-factoid-itb/train_preprocess.csv" -P "data/"

# # download validation dataset
# !wget "https://raw.githubusercontent.com/indobenchmark/indonlu/master/dataset/facqa_qa-factoid-itb/valid_preprocess.csv" -P "data/"

In [2]:
data_files = {"train": 'data/train_preprocess.csv', "val": 'data/valid_preprocess.csv'}

dataset = load_dataset('csv', data_files=data_files)

## Set config

In [3]:
# we will finetuning using indobert-base-uncased from indoLEM
model_checkpoint = "cahya/distilbert-base-indonesian" 
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
batch_size = 4

In [4]:
# set config when tokenizing
encoder_max_len = 512
doc_stride = 128
pad_on_right = tokenizer.padding_side == "right"

## Preprocess data

In [5]:
# encode function for encoding data
def encode(example, encoder_max_len=encoder_max_len):
    
    texts = copy.copy(example['passage'])
    questions= copy.copy(example['question'])
    answers = copy.copy(example['seq_label'])
    answers_text = [None for i in range(len(texts))]
    
    # since the data type in string, we need to convert the data into list
    for i in range(len(texts)):
        t = texts[i].strip("']['").split("', '")
        a = answers[i].strip("']['").split("', '")
        q = questions[i].strip("']['").split(", ")
        q = [b.strip('\'"') for b in q]
        
        if len(t)!=len(a):
            t = texts[i].strip("']['").split(", ")
            t_swap = []
            for b in t:
                if b[0] == '"':
                    t_swap.append(b.strip('"'))
                else:
                    t_swap.append(b.strip("\'"))
            t = t_swap
            
        assert len(t)==len(a)
        
        answers[i] = a
        texts[i] = t
        questions[i] = q
        answers_text[i] = " ".join(list(np.array(t)[np.array(a) != 'O']))
        

    # encode after converting the data
    encoder_inputs = tokenizer(questions, texts, is_split_into_words=True, truncation="only_second", max_length=encoder_max_len, padding=False, 
                               return_overflowing_tokens=True, return_offsets_mapping=True, stride=doc_stride)

    input_ids = encoder_inputs['input_ids']
    input_attention = encoder_inputs['attention_mask']
    offset_mapping = encoder_inputs.pop("offset_mapping") 

    # get the start and end index position of the answers for the questions  
    start_answer_token_positions = []
    end_answer_token_positions = []
    
    for i in range(len(texts)):
        sequence_ids = encoder_inputs.sequence_ids(i)
        
        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1
            
        token_end_index = len(input_ids[i]) - 1
        while sequence_ids[token_end_index] != 1 :
            token_end_index -= 1
            
        start_token_answer = 0
        while answers[i][start_token_answer] == 'O':
            if offset_mapping[i][token_start_index + start_token_answer +1][0] == 0:
                start_token_answer += 1
            else:
                token_start_index += 1
        
        start_answer_token_positions.append(token_start_index + start_token_answer)
        
        end_token_answer = len(answers[i]) -1
        while answers[i][end_token_answer] == 'O':
            if offset_mapping[i][token_end_index][0] == 0:
                end_token_answer -= 1
                token_end_index -= 1
            else:
                token_end_index -= 1
                
        end_answer_token_positions.append(token_end_index + 1)
    
    outputs = {'input_ids':input_ids, 'attention_mask': input_attention, 
               "start_positions": start_answer_token_positions, "end_positions": end_answer_token_positions}
    
    return outputs

In [6]:
# preprocess dataset, encode function will be applied on the fly
dataset.set_transform(encode)

In [7]:
# check sample of tokenized data
print(dataset['train'][:2])

{'input_ids': [[3, 2560, 5289, 1510, 2793, 5513, 4136, 1930, 8839, 1495, 4351, 1990, 5271, 32, 1, 4920, 6204, 1519, 3564, 4432, 3735, 4536, 22855, 7727, 9647, 15, 3109, 1049, 1013, 16, 2482, 23420, 10576, 1489, 15, 2560, 15335, 18229, 1510, 2997, 15, 2793, 5513, 4136, 1930, 8839, 1495, 4351, 1990, 5271, 17, 1], [3, 1495, 2408, 2369, 5944, 15, 3521, 13606, 8680, 4767, 11432, 6451, 1941, 6398, 3347, 16, 1571, 1, 2079, 1582, 4226, 16, 1571, 1495, 1750, 10107, 16, 10107, 2218, 1887, 2316, 13141, 13606, 15, 1971, 3521, 13606, 8680, 4767, 11432, 2408, 6760, 1509, 3521, 7263, 16238, 17, 3521, 13606, 8680, 4767, 11432, 6451, 1495, 2408, 6760, 1509, 3521, 7263, 16238, 1495, 10813, 7571, 1007, 17, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

## Prepare model

We will finetuning the model using the Trainer class from transformers library

In [8]:
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: cahya/distilbert-base-indonesian
Key                     | Status     | 
------------------------+------------+-
vocab_projector.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
# set config argument for Trainer object
args = TrainingArguments(
    output_dir='./results', 
    eval_strategy = "epoch",
    save_strategy = "epoch",
    logging_strategy = "epoch",
    remove_unused_columns=False, 
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=2,
    weight_decay=0.01,
    gradient_accumulation_steps=4,
    average_tokens_across_devices=False,
    load_best_model_at_end = True,
    metric_for_best_model = 'eval_loss',
    save_total_limit = 1,    
)

In [10]:
# for more info check this https://huggingface.co/transformers/main_classes/data_collator.html
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [11]:
# create Trainer object
trainer = Trainer(
    model,
    args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=data_collator,
)

## Finetuning the model

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.524030,1.573096
2,1.231119,1.235404


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=312, training_loss=1.8775746761224208, metrics={'train_runtime': 115.9413, 'train_samples_per_second': 43.039, 'train_steps_per_second': 2.691, 'total_flos': 122814169211916.0, 'train_loss': 1.8775746761224208, 'epoch': 2.0})

In [13]:
# # save the model
# trainer.save_model('best_qa_model')

In [14]:
# test to make sure that the best model is loaded at the end
trainer.evaluate()

Training Loss,Validation Loss,Epoch
1.231119,1.235404,2


{'eval_loss': 1.2354040145874023}

## Using the saved model

In [15]:
# get device function
def get_default_device():
    """Pick GPU if available, else CPU"""
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

In [16]:
# set the device to put the best model into
device = get_default_device()
print(device)

cuda


In [17]:
# load the best model
best_model_checkpoint = "results/checkpoint-312"
best_model = AutoModelForQuestionAnswering.from_pretrained(best_model_checkpoint)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [18]:
# set the model to device, using cuda for faster calculation
model.to(device)

DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

In [24]:
# convert raw input into feature vector
def prepare_features(example):

    text = example['passage']
    question = example['question']
        
    # Tokenize our examples with truncation and maybe padding, but keep the overflows using a stride. This results
    # in one example possible giving several features when a context is long, each of those features having a
    # context that overlaps a bit the context of the previous feature.

    tokenized_examples = tokenizer(question, text, is_split_into_words=False, truncation="only_second", 
                                   max_length=encoder_max_len, padding=False, 
                                   return_overflowing_tokens=True, return_offsets_mapping=True, stride=doc_stride)

    # Since one example might give us several features if it has a long context, we need a map from a feature to
    # its corresponding example. This key gives us just that.
    #sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")

    # Grab the sequence corresponding to that example (to know what is the context and what is the question).
    context_index = 1 if pad_on_right else 0

    return tokenized_examples

In [25]:
# example of input
ex_input={'passage': 'Hal ini banyak dicari tahu masyarakat sejak Presiden Joko Widodo memimpin Upacara Penetapan Komponen Cadangan (Komcad) pada hari ini (7/10/2021) di Pusdiklatpassus, Bandung, Jawa Barat.', 
          'question': 'Dimanakah Upacara Penetapan Komponen Cadangan diadakan ?'}

In [26]:
# process input into feature vector
feature_input = prepare_features(ex_input)

In [31]:
feature_input

{'input_ids': [[3, 3329, 5944, 4155, 11425, 5357, 8333, 3587, 32, 1, 1938, 1542, 1848, 15408, 4796, 2246, 2079, 2601, 10492, 12328, 4258, 4155, 11425, 5357, 8333, 11, 1736, 2586, 1006, 12, 1535, 1994, 1542, 11, 26, 18, 2178, 18, 16564, 12, 1495, 20729, 5250, 5694, 4325, 15, 3786, 15, 2326, 2070, 17, 1]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'offset_mapping': [[(0, 0), (0, 6), (6, 9), (10, 17), (18, 27), (28, 36), (37, 45), (46, 54), (55, 56), (0, 0), (0, 3), (4, 7), (8, 14), (15, 21), (22, 26), (27, 37), (38, 43), (44, 52), (53, 57), (58, 64), (65, 73), (74, 81), (82, 91), (92, 100), (101, 109), (110, 111), (111, 114), (114, 116), (116, 117), (117, 118), (119, 123), (124, 

In [27]:
# function for prediction
def predict_answers(features, max_answer_length = 30, n_best_size = 20):
     

    best_answers =[]
        
    attention_mask = torch.tensor(features['attention_mask']).to(torch.int64)
    input_ids = torch.tensor(features['input_ids']).to(torch.int64)
    token_type_ids = torch.tensor(features['token_type_ids']).to(torch.int64)

    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    token_type_ids = token_type_ids.to(device)

    with torch.no_grad():
        output = model(attention_mask = attention_mask, input_ids = input_ids, token_type_ids=token_type_ids)


    start_logits = output.start_logits[0].cpu().numpy()
    end_logits = output.end_logits[0].cpu().numpy()
    offset_mapping = features["offset_mapping"][0]

    context = features['input_ids'][0]

    # Gather the indices the best start/end logits:
    start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
    end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()

    valid_answers = []
    for start_index in start_indexes:
        for end_index in end_indexes:
            # Don't consider out-of-scope answers, either because the indices are out of bounds or correspond
            # to part of the input_ids that are not in the context.
            if (
                start_index >= len(offset_mapping)
                or end_index >= len(offset_mapping)
                or offset_mapping[start_index] is None
                or offset_mapping[end_index] is None
            ):
                continue
            # Don't consider answers with a length that is either < 0 or > max_answer_length.
            if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                continue
            #if start_index <= end_index: # We need to refine that test to check the answer is inside the context
            #start_char = offset_mapping[start_index][0]
            #end_char = offset_mapping[end_index][1]
            valid_answers.append(
                {
                    "score": start_logits[start_index] + end_logits[end_index],
                    "text": tokenizer.decode(context[start_index: end_index]),
                    "start_idx": start_index,
                    "end_idx": end_index
                }
            )

    valid_answers = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[:n_best_size]
    
    try:
        best_answers.append(valid_answers[0])
    except:
        print(i)
        print(idx)
        print(valid_answers)
        print(start_indexes)
        print(end_indexes)

    #return best_answers
    return valid_answers

In [28]:
# predict funtion return all possible answer sorted by its score, answer with the biggest score is the top answer
predict_answers(feature_input)

[{'score': np.float32(9.855064),
  'text': 'komcad ) pada hari ini ( 7 / 10 / 2021 ) di pusdiklatpassus, bandung, jawa barat',
  'start_idx': 26,
  'end_idx': 50},
 {'score': np.float32(9.507227),
  'text': 'pusdiklatpassus, bandung, jawa barat',
  'start_idx': 41,
  'end_idx': 50},
 {'score': np.float32(9.263855),
  'text': '7 / 10 / 2021 ) di pusdiklatpassus, bandung, jawa barat',
  'start_idx': 34,
  'end_idx': 50},
 {'score': np.float32(8.755902),
  'text': 'komcad ) pada hari ini ( 7 / 10 / 2021',
  'start_idx': 26,
  'end_idx': 39},
 {'score': np.float32(8.53566),
  'text': 'komcad',
  'start_idx': 26,
  'end_idx': 29},
 {'score': np.float32(8.164693),
  'text': '7 / 10 / 2021',
  'start_idx': 34,
  'end_idx': 39},
 {'score': np.float32(7.1276894),
  'text': 'komcad )',
  'start_idx': 26,
  'end_idx': 30},
 {'score': np.float32(7.029807),
  'text': 'komcad ) pada hari ini ( 7 / 10 / 2021 )',
  'start_idx': 26,
  'end_idx': 40},
 {'score': np.float32(7.017171),
  'text': 'komcad )

In [42]:
# print the best answer
best_answer = predict_answers(feature_input)[0]['text']
print(best_answer)

7 / 10 / 2021 ) di pusdiklatpassus, bandung, jawa barat
